In [38]:
import sys
import os

# Make project root importable so we can use custom matroid functions
current_dir = os.getcwd()
parent_dir = os.path.dirname(current_dir)
sys.path.insert(0, parent_dir)

import sage.all
from sage.all import *

from custom_matroid_functions import *

# Laurent polynomial ring in x
P = LaurentPolynomialRing(ZZ, 'x')
x = P.gen()

In [39]:
# Helper: choose a test matroid and an element e

def make_test_matroid(k, n=5):
    """Return (M, e) where M is K_n and e is a distinguished element."""
    #M = matroids.CompleteGraphic(n)
    M = matroids.Uniform(k, n)
    e = list(M.groundset())[0]
    return M, e

# Short aliases to polynomials we need
chow = chow_polynomial
chibar = reduced_characteristic_polynomial
kl_poly = lambda M: M.lattice_of_flats().kazhdan_lusztig_polynomial()(x)

M, e = make_test_matroid(3, 5)
print("Ground set:", M.groundset())
print("Deleting element:", e)

Ground set: frozenset({0, 1, 2, 3, 4})
Deleting element: 0


In [40]:
def flat_division(M, e):
    """Partition flats according to whether they contain e and whether adding/removing e stays flat."""
    L = M.lattice_of_flats()
    flat_div = {'S': set(), 'T': set(), 'A': set(), 'B': set()}
    for F in L:
        if e in F:
            if F - {e} in L:
                flat_div['T'].add(F)
            else:
                flat_div['A'].add(F)
        else:
            if F.union({e}) in L:
                flat_div['S'].add(F)
            else:
                flat_div['B'].add(F)
    return flat_div


def convolve(f, g, M):
    E = M.groundset()
    flats = list(M.lattice_of_flats())
    total = P(0)
    for F in flats:
        total += f(M.delete(E - F))(x) * g(M.contract(F))(x)
    return total


def rev(f):
    return lambda M: f(M)(x**-1) * x**M.rank()


def right_aug_poly(M):
    """Right augmented Chow polynomial F(M) = H * f_rev."""
    return convolve(chow, rev(kl_poly), M)


print("right_aug_poly(M) =", right_aug_poly(M))

right_aug_poly(M) = 1 + 21*x + 21*x^2 + x^3


In [41]:
# Quick sanity check on basic invariants

print("rank of M:", M.rank())
print("chow polynomial of M:", chow_polynomial(M))
print("reduced characteristic polynomial:", reduced_characteristic_polynomial(M))

rank of M: 3
chow polynomial of M: x^2 + 11*x + 1
reduced characteristic polynomial: x^2 - 4*x + 6


In [42]:
def deletion_sum_right_aug(M, e):
    flat_div = flat_division(M, e)
    S_flats = flat_div['S']

    deleted_poly = right_aug_poly(M.delete(e))
    total = deleted_poly

    for F in S_flats:
        poly1 = chow_polynomial(M.contract(F.union({e})))
        poly2 = right_aug_poly(M.delete(M.groundset() - F))
        term = x * poly1 * poly2
        total += term

    return total


def check_deletion_formula(M, e):
    """Compare F(M) with the deletion-type expression and report the difference."""
    og_poly = right_aug_poly(M)
    expr = deletion_sum_right_aug(M, e)
    print("F(M)             =", og_poly)
    print("F(M.delete(e))   =", right_aug_poly(M.delete(e)))
    print("deletion expr    =", expr)
    print("difference       =", og_poly - expr)
    return og_poly, expr


check_deletion_formula(M, e)


F(M)             = 1 + 21*x + 21*x^2 + x^3
F(M.delete(e))   = 1 + 13*x + 13*x^2 + x^3
deletion expr    = 1 + 18*x + 18*x^2 + x^3
difference       = 3*x + 3*x^2


(1 + 21*x + 21*x^2 + x^3, 1 + 18*x + 18*x^2 + x^3)